In [1]:
%load_ext autoreload
%autoreload 2

# Compute coupling for DRN models

We will utilize Enesmble Corpula Coupling (ECC) and a Gaussian Corpula Approach (GCA)

## ECC

Steps:
1. **Univariate postprocessing:** Obtain univariately postprocessed marginal distibutions for each margin. Here we use location and variable as the margins. This is simply the ouput of a model that inherits from `DistibutionRegression`.

2. **Quantization:** Draw samples from each marginal predictive distibution. The samples should have the same size as the raw ensemble $M$. We sample at equidistant quantiles $\frac{1}{M+1}, \dots, \frac{M}{M+1}$. Note that for the wind speed the distribution is a truncated normal. This yields the samples $\tilde{x}^l_1, \dots, \tilde{x}^l_M$, where $l \in L=I \times J$, $I$ is the set of stations and $J$ is the set of features (variables).

3. **Ensemble Reordering:** Reorder based on the rank structure of the original ensemble with possible ties resolved at random. For each margin $l$, the order statistics of the raw ensemble values $x_{(1)}^l, \dots, x_{(M)}^l$ introduce a permutation $\sigma_l$ of the integers ${1, \dots, M}$ defined by $\sigma_l(m)=\operatorname{rk}(x_m^l)$ for $m = 1, \dots, M$. Ties in the rank are resolved at random. The respecitve margin of the ECC postprocessed ensemble is then given by $\hat{x}=\tilde{x}_{\sigma_l(1)}, \dots, \tilde{x}_{\sigma_l(M)}$.

## GCA
First of all some problems with this approach:
- In the common case in which the dimension L (= all station, variable, lead time pairs) of the output quantity is huge and no specific structure can be exploited, parametric methods are bound to fail.

Steps:
1. **Transform to latent Gaussian:** Transform a set of past observarions into latent standard Gaussian observations via $$\tilde{y}^l=\Phi^{-1}(F_\theta^l(y^l))$$ where $F_\theta^d$ is the distribution for each margin.
2. **Sampling:** Calculate $\Sigma$ based on $\tilde{y}^l$ obtained in Step 1. Then sample from $N(0,\Sigma)$ to obtain $Z_1, \dots, Z_M$.
3. **Reverse Transform:** Transform back to original space: $$X_m^l=(F_\theta^l)^{-1}(\Phi(Z_m^l))$$



In [ ]:
import importlib
from collections import defaultdict

import hydra
import lightning as L
import numpy as np
import torch
import xarray as xr
from einops import rearrange
from omegaconf import DictConfig
from scipy.stats import multivariate_normal, norm, truncnorm
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import FC_VARS, FORECAST_ENS_PATH, MISSING_DAYS, OBSERVATIONS_FLAT_PATH
from genpp.data.utils import get_time_intersection
from genpp.eval.utils import log_scores
from genpp.models.loss import EnergyScore

try:
    register_resolvers()
except Exception:
    pass

In [ ]:
MODEL_ID = "4pgm4fll"
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

# Load model and dataloader to get predictions
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
    # Do not shuffle any dataloader
    cfg.data.module.dataloader_config.train.shuffle = False
    cfg.data.module.dataloader_config.val.shuffle = False
    cfg.data.module.dataloader_config.test.shuffle = False
    datamodule = hydra.utils.instantiate(cfg.data.module)
    model_class = cfg.model._target_.split(".")[-1]
datamodule.prepare_data()
datamodule.setup(stage="validate")

In [ ]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

In [ ]:
trainer = L.Trainer(logger=False)
predictions = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)

In [ ]:
def stack_predictions(predictions) -> list[dict[str, torch.Tensor]]:
    n_dicts = len(predictions[0])
    merged = [defaultdict(list) for _ in range(n_dicts)]

    # accumulate tensors per key
    for batch in predictions:
        for i, d in enumerate(batch):
            for k, v in d.items():
                merged[i][k].append(v)

    # concatenate along first dimension
    return [{k: torch.cat(vs, dim=0) for k, vs in m.items()} for m in merged]


def predictions_to_dataarray(obs_da, preds):
    """
    obs_da: original xarray.DataArray of observations
    preds: list of dicts, each dict corresponds to one feature
           dict values are torch.Tensors with shape (time, longitude, latitude)
    """
    features = obs_da.coords["feature"].values  # ["2m_temperature", "10m_wind_speed"]

    new_data = []
    new_feature_names = []

    for feat_name, pred_dict in zip(features, preds):
        for param_name, tensor in pred_dict.items():
            # ensure numpy
            arr = tensor.detach().cpu().numpy()
            new_data.append(arr)
            new_feature_names.append(f"{feat_name}_{param_name}")

    # stack along new "feature" axis
    data = np.stack(new_data, axis=1)  # (time, new_features, longitude, latitude)

    # build DataArray
    return xr.DataArray(
        data,
        coords={
            "time": obs_da.coords["time"],
            "feature": new_feature_names,
            "longitude": obs_da.coords["longitude"],
            "latitude": obs_da.coords["latitude"],
        },
        dims=("time", "feature", "longitude", "latitude"),
    )

In [ ]:
# Next step get this in order to a nice datastructure
stacked = stack_predictions(predictions)
val_split = hydra.utils.instantiate(cfg.data.splits.val)
y_da = xr.open_dataarray(OBSERVATIONS_FLAT_PATH).sel(time=val_split)
predictions_xr = predictions_to_dataarray(y_da, stacked)

## ECC Step 2: Quantization
Draw equally spaced samples from the distributions where for the temperature the distibution is $N(\mu, \sigma)$.
And for the wind speed it is a truncated normal with bounds at 0 and $\infty$.
Since the ensemble has 50 members we will draw 50 samples.

In [12]:
def quantile_samples(pred_da, M=50):
    """
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']
    M: number of quantile samples

    Returns: xarray.DataArray with dims (time, feature, samples, longitude, latitude)
    """
    time = pred_da.coords["time"]
    longitude = pred_da.coords["longitude"]
    latitude = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # quantile positions
    qs = np.linspace(1 / (M + 1), M / (M + 1), M)

    # temperature (normal distribution)
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values  # (time, lon, lat)
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    temp_samples = norm.ppf(
        qs[:, None, None, None], loc=mu_temp[None, ...], scale=sigma_temp[None, ...]
    )
    # (M, time, lon, lat)

    # wind speed (truncated normal, bounds [0, ∞))
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    # From the Docs:
    # If a_trunc and b_trunc are the abscissae at which we wish to truncate the distribution
    # (as opposed to the number of standard deviations from loc),
    # then we can calculate the distribution parameters a and b as follows:
    # a, b = (a_trunc - loc) / scale, (b_trunc - loc) / scale

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    wind_samples = truncnorm.ppf(
        qs[:, None, None, None],
        a[None, ...],
        b,
        loc=mu_wind[None, ...],
        scale=sigma_wind[None, ...],
    )
    # (M, time, lon, lat)

    # stack along feature axis
    data = np.stack([temp_samples, wind_samples], axis=1)  # (M, feature, time, lon, lat)

    # reorder to (time, feature, sample, lon, lat)
    data = rearrange(data, "sample feature time lon lat -> time feature sample lon lat")

    return xr.DataArray(
        data,
        coords={
            "time": time,
            "feature": features,
            "sample": np.arange(M),
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("time", "feature", "sample", "longitude", "latitude"),
    )

In [13]:
pred_samples = quantile_samples(predictions_xr, M=50)

## ECC Step 3: Ensemble Reordering

Now load the Ensemble predictions and determine the rank ordering and use this to reshuffle the predicted samples

In [14]:
ens = xr.open_dataset(FORECAST_ENS_PATH)
ens = ens.sel(time=~ens.time.isin(MISSING_DAYS))
ens = ens.assign_coords(prediction_time=ens.time + ens.prediction_timedelta).swap_dims(
    {"time": "prediction_time"}
)
times = get_time_intersection(ens, pred_samples)

ens = ens.sel(prediction_time=times)[FC_VARS]
ens = (
    ens.to_array()
    .rename({"variable": "feature"})
    .transpose("prediction_time", "feature", "number", "longitude", "latitude")
)

In [15]:
# Random tie breaking is done by noising the data slightly to avoid ties
# Theoretically ties would still be possible, but insanely unlikely
rng = np.random.default_rng(seed=42)
noise = rng.uniform(low=-1e-10, high=1e-10, size=ens.shape)
ens_noised = ens + noise

ens_ranked = ens_noised.rank(dim="number") - 1  # -1 to get ranks from 0 to 49
ens_ranked = ens_ranked.astype(np.int32)

In [16]:
# Get the same dims so that the reodering works
indexer = ens_ranked.drop_vars("time", errors="ignore")
indexer = indexer.rename({"prediction_time": "time", "number": "sample"})
# Drop this var since for every sample we want the indexing array
indexer = indexer.drop_vars("sample", errors="ignore")

# Use advanced indexing
# Note to me: I tested this and it works correctly
pred_samples_reordered = pred_samples.isel(sample=indexer)

## Now compute the energy scores
For this we need to convert the underlying arrays to tensors and collect the scores afterwards
We can loop over the (batched) days to calculate these scores

In [17]:
es = EnergyScore(clamp=False)

In [ ]:
pred_samples_reordered = pred_samples_reordered.transpose(
    "time", "sample", "feature", "longitude", "latitude"
)

times = y_da.time
scores = []
for t in tqdm(times):
    curr_obs = y_da.sel(time=t).to_numpy()
    curr_pred = pred_samples_reordered.sel(time=t).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)
    score = es(pred_t, obs_t)
    scores.append(score)
scores = torch.cat(scores, dim=0)
print(f"Mean score for t2m and 10m wind speed: {scores.mean(dim=0)}")
log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=scores.mean(dim=0),
)
# TODO: compute other scores

100%|██████████| 730/730 [00:01<00:00, 490.33it/s]

Mean score for t2m and 10m wind speed: tensor([26.5394, 20.7180])


## GCA Postprocessing
### Step 1
Transforming to latent distribution to estimate $\Sigma$
What we need for that:
- Observations from the train Set
- Forecasts from the train set

In [19]:
train_split = hydra.utils.instantiate(cfg.data.splits.train)
y_train = xr.open_dataarray(OBSERVATIONS_FLAT_PATH).sel(time=train_split)

datamodule.setup(stage="fit")
predictions_train = trainer.predict(model, datamodule.train_dataloader(), return_predictions=True)
predictions_train = stack_predictions(predictions_train)
predictions_train = predictions_to_dataarray(y_train, predictions_train)

/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Predicting DataLoader 0: 100%|██████████| 35/35 [00:00<00:00, 42.81it/s]


In [ ]:
def transform_to_latent_gaussian(obs_da, pred_da, eps=1e-9):
    """
    Transform observations into latent Gaussian space.

    obs_da: xarray.DataArray with dims (time, feature, longitude, latitude)
            containing observed values. Feature names must match:
            ["2m_temperature", "10m_wind_speed"]

    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray with dims (time, feature, longitude, latitude)
             containing latent Gaussian transformed observations.
    """

    time = obs_da.coords["time"]
    longitude = obs_da.coords["longitude"]
    latitude = obs_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # ---- Temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values
    obs_temp = obs_da.sel(feature="2m_temperature").values

    # Compute CDF then transform
    cdf_temp = norm.cdf(obs_temp, loc=mu_temp, scale=sigma_temp)
    # Add clamping to avoid infs
    cdf_temp = np.clip(cdf_temp, eps, 1 - eps)
    latent_temp = norm.ppf(cdf_temp)

    # ---- Wind speed (truncated normal, [0, inf)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values
    obs_wind = obs_da.sel(feature="10m_wind_speed").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    cdf_wind = truncnorm.cdf(obs_wind, a=a, b=b, loc=mu_wind, scale=sigma_wind)
    # Add clamping to avoid infs
    cdf_wind = np.clip(cdf_wind, eps, 1 - eps)
    latent_wind = norm.ppf(cdf_wind)

    # ---- Stack back into xarray ----
    data = np.stack([latent_temp, latent_wind], axis=1)  # (time, feature, lon, lat)

    return xr.DataArray(
        data,
        coords={
            "time": time,
            "feature": features,
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("time", "feature", "longitude", "latitude"),
    )

In [125]:
latent = transform_to_latent_gaussian(y_train, predictions_train)

In [ ]:
flat = latent.stack(space=("feature", "longitude", "latitude"))
# shape: (time, space)
X = flat.values
# TODO check what values are used for the computation
# compute covariance matrix across space
Sigma = np.cov(X, rowvar=False)  # shape (space, space)

## Step 2: Samples from the latent Space

In [180]:
t_steps = len(predictions_xr.time)
n_samples = 50

latent_samples = multivariate_normal.rvs(
    mean=np.zeros(Sigma.shape[0]),
    cov=Sigma,  # type: ignore
    size=(t_steps, n_samples),  # type: ignore
)
latent_samples = latent_samples.reshape(t_steps, n_samples, *y_da.shape[1:])

## Step 3: Transform back to the initial space using the predicted Distributions

In [192]:
def inverse_transform_latent(latent_samples, pred_da, eps=1e-9):
    """
    Transform latent Gaussian samples back to original space.

    latent_samples: np.array of shape (time, n_samples, feature, lon, lat)
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray of shape (time, sample, feature, lon, lat)
    """
    time = pred_da.coords["time"]
    lon = pred_da.coords["longitude"]
    lat = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    n_time, n_samples, n_feature, n_lon, n_lat = latent_samples.shape

    # Transform to uniform [0,1] via standard normal CDF
    uniform_samples = norm.cdf(latent_samples)

    # Clamp the uniform samples to avoid issues with PPF
    uniform_samples = np.clip(uniform_samples, eps, 1 - eps)

    # Allocate array for original space
    orig_samples = np.zeros_like(uniform_samples)

    # ---- 2m temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    orig_samples[:, :, 0, :, :] = norm.ppf(
        uniform_samples[:, :, 0, :, :],
        loc=mu_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
        scale=sigma_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
    )

    # ---- 10m wind speed (truncated normal, [0, ∞)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf

    orig_samples[:, :, 1, :, :] = truncnorm.ppf(
        uniform_samples[:, :, 1, :, :],
        a=a[:, None, :, :],
        b=b,
        loc=mu_wind[:, None, :, :],
        scale=sigma_wind[:, None, :, :],
    )

    return xr.DataArray(
        orig_samples,
        coords={
            "time": time,
            "sample": np.arange(n_samples),
            "feature": features,
            "longitude": lon,
            "latitude": lat,
        },
        dims=("time", "sample", "feature", "longitude", "latitude"),
    )


gca_preds = inverse_transform_latent(latent_samples, predictions_xr)

### Lastly compute the energy scores

In [ ]:
times = y_da.time
scores = []
for t in tqdm(times):
    curr_obs = y_da.sel(time=t).to_numpy()
    curr_pred = gca_preds.sel(time=t).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)
    score = es(pred_t, obs_t)
    scores.append(score)
scores = torch.cat(scores, dim=0)

print(f"Mean score for t2m and 10m wind speed: {scores.mean(dim=0)}")

# Log the scores to a file in the MODEL_PATH folder
log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=scores.mean(dim=0),
)

100%|██████████| 730/730 [00:01<00:00, 529.93it/s]

Mean score for t2m and 10m wind speed: tensor([26.7553, 20.8471])
